# Day 2 · 01 · Power BI Semantic Model on Databricks

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

This notebook prepares the Power BI handoff. The model stays in Databricks;
Power BI receives clean Gold objects, a small number of measures, and a clear
contract.

## Business Scenario

RetailHub has trusted Gold objects in Databricks. The next decision is how Power BI should consume them: which object supports the summary page, which object supports drill-through, and when Import mode or DirectQuery makes sense.

The goal is to keep Power BI focused on relationships, a small number of measures and report interactions, while Databricks owns reusable data logic.

## Learning Objectives

By the end of this notebook you can:

- choose Databricks objects for Power BI visuals,
- decide between Import mode and DirectQuery,
- prepare an incremental-refresh view,
- recognize what folds into SQL in Power Query and what does not,
- use Performance Analyzer to tell a Databricks-side slowdown from a
  Power BI/DAX-side one,
- fill a BI contract,
- explain when AI/BI Dashboards can complement Power BI.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.kpi_monthly",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 1. Semantic model principle

Model in Databricks; keep Power BI thin.

Power BI should contain:

- relationships,
- report interactions,
- a small number of measures,
- presentation logic.

Databricks should contain:

- joins,
- grain control,
- KPI logic,
- reusable aggregates,
- data-quality gates,
- refresh-ready objects.

## 2. Reporting table vs reporting view

| Pattern | Use when | Risk |
|---|---|---|
| Gold table | many users query same result | refresh must be orchestrated |
| Gold view | logic changes often, row count small | every query recomputes |
| Gold aggregate | summary visuals repeat often | can hide detail if overused |
| Power BI measure | presentation-only calculation | logic can fragment across reports |

## 3. Build BI-ready objects

`fact_sales_dashboard_monthly` already supports summary pages. Now create a
bounded detail view for drill-through and incremental refresh.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.v_fact_sales_incremental
COMMENT 'Line-grain BI drill-through view bounded to recent data and suitable for Power BI incremental refresh predicates.'
AS
SELECT *
FROM {GOLD}.fact_sales_dashboard
WHERE order_date >= add_months(current_date(), -24)
""")

spark.sql(f"""
SELECT MIN(order_date) AS min_date, MAX(order_date) AS max_date, COUNT(*) AS rows
FROM {GOLD}.v_fact_sales_incremental
""").show()

In [ ]:
spark.sql(f"""
SELECT
  'summary_page' AS report_area,
  'gold.fact_sales_dashboard_monthly' AS source_object,
  'year_month x customer_region x category x channel' AS grain,
  COUNT(*) AS rows
FROM {GOLD}.fact_sales_dashboard_monthly
UNION ALL
SELECT
  'drill_through' AS report_area,
  'gold.v_fact_sales_incremental' AS source_object,
  'order line' AS grain,
  COUNT(*) AS rows
FROM {GOLD}.v_fact_sales_incremental
""").show(truncate=False)

## 4. Power BI model design

![Star schema vs flat table](../../assets/images/star_schema_vs_flat_table.png)

Recommended dataset:

- summary page: `gold.fact_sales_dashboard_monthly`,
- drill-through: `gold.v_fact_sales_incremental`,
- slicers: `gold.dim_date`, `gold.dim_customer`, `gold.dim_product`,
- KPI cards: `gold.kpi_monthly` or summary aggregate.

Power BI measures should be thin. Example measures:

```DAX
Revenue = SUM(fact_sales_dashboard_monthly[revenue])
Gross Margin = SUM(fact_sales_dashboard_monthly[gross_margin])
Margin Rate % = DIVIDE([Gross Margin], [Revenue])
Completed Orders = SUM(fact_sales_dashboard_monthly[completed_orders])
```

## 5. Import mode path

Import is the baseline for most executive KPI dashboards.

| Why Import | RetailHub answer |
|---|---|
| Dashboard does not need second-level freshness | daily refresh is enough |
| Many users open the same summary page | import avoids query fan-out |
| Summary objects are small | monthly aggregate is compact |
| Cost should be predictable | refresh job cost is schedulable |

## 6. Incremental refresh

![Incremental refresh range](../../assets/images/incremental_refresh_range.png)

Power BI incremental refresh uses two parameters:

- `RangeStart`
- `RangeEnd`

The SQL pattern must be half-open:

```sql
WHERE order_date >= RangeStart
  AND order_date <  RangeEnd
```

In [ ]:
range_start = "2025-01-01"
range_end = "2025-04-01"

spark.sql(f"""
SELECT
  COUNT(*) AS rows_in_window,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '{range_start}'
  AND order_date <  DATE '{range_end}'
""").show()

## 7. DirectQuery / live path

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

DirectQuery is powerful, but it moves work from refresh time to user
interaction time. Treat it as an architectural decision, not just a connection
mode checkbox.

| Question | Import | DirectQuery |
|---|---|---|
| Freshness | scheduled | live |
| Query execution | refresh time | every visual interaction |
| Cost driver | planned refresh | clicks x visuals x users |
| Best object | aggregate table | narrow Gold table/view |
| RetailHub default | yes | exception only |

## 8. Query folding — what folds vs what breaks it

Query folding is what makes DirectQuery (and Import refresh queries) cheap:
Power Query pushes its steps down into a single SQL query that Databricks SQL
Warehouse executes, instead of pulling raw rows and transforming them
client-side.

| Power Query step | Folds into SQL? | Why |
|---|---|---|
| filter rows (`Table.SelectRows`) | yes | becomes `WHERE` |
| remove/select columns | yes | becomes column list in `SELECT` |
| group by / aggregate | yes | becomes `GROUP BY` |
| sort | yes | becomes `ORDER BY` |
| merge with another folded query on the same source | usually | becomes a `JOIN` |
| custom M function / `Table.AddColumn` with complex logic | often **not** | runs client-side after the SQL result comes back |
| merging queries from two different data sources | **no** | nothing to fold into, both sides must materialize first |
| changing a column's case/type with non-trivial M logic | sometimes not | depends on whether Power Query can express it in SQL |

How to check folding in Power Query: right-click the last applied step — if
**"View Native Query"** is available (not greyed out), the query still folds.
The first step where it greys out is where folding broke; everything before
that step ran in Databricks SQL, everything after runs in Power BI's engine
on rows already pulled across the wire.

Practical rule for this course: keep transformation logic in Gold (where it
always folds, because it is already SQL) and keep Power Query close to a
plain `SELECT` from a Gold object. Push complex logic upstream, not into M.

## `[UI DEMO]` Power BI connector

![Power BI connection walkthrough](../../assets/images/powerbi_connection_walkthrough.png)

Show:

1. SQL Warehouse connection details.
2. Server hostname.
3. HTTP path.
4. Power BI Databricks connector.
5. Import vs DirectQuery selection.
6. Query History after interacting with the report.

If Power BI Desktop is not available, narrate this with the screenshot and the
decision table above.

## 9. Performance Analyzer orientation

When a visual is slow, the first question is: is this a Databricks problem or
a Power BI problem? `day2_02` covers the Databricks side (Query History,
Query Profile). This is the Power BI-side counterpart.

`[UI DEMO]` In Power BI Desktop, open **View → Performance Analyzer**, click
**Start Recording**, then refresh a visual:

| Performance Analyzer row | What it measures | If it's high |
|---|---|---|
| DAX query | time Power BI's engine spent computing the measure | check the measure/DAX, not Databricks |
| Visual display | time spent rendering the visual itself | too many data points / visual complexity |
| Other (incl. query to data source) | time waiting on the SQL Warehouse for DirectQuery/refresh | this is where Query History/Query Profile come in |

If "Other" dominates, go to Databricks Query History for that time window and
read the Query Profile, same as in `day2_02`. If "DAX query" dominates, the
fix is in the measure or model, not in Databricks SQL.

Rule of thumb: Performance Analyzer tells you *which side* is slow before you
spend time optimizing the wrong layer.

## 10. BI contract

Use `docs/templates/bi-contract.md`.

| Area | Decision |
|---|---|
| Summary source | `gold.fact_sales_dashboard_monthly` |
| Detail source | `gold.v_fact_sales_incremental` |
| KPI source | `gold.kpi_monthly` |
| Grain summary | month x region x category x channel |
| Grain detail | order line |
| Baseline mode | Import |
| DirectQuery exception | only for small operational live pages |
| Refresh | daily after Gold refresh |
| Owner | Sales Ops + Data team |
| Known limitation | dashboard is not second-level real time |

## 11. Optional bonus: AI/BI Dashboards and Genie

AI/BI Dashboards are useful when:

- a team wants a fast Databricks-native dashboard,
- Power BI Desktop is not available during training,
- analysts want to test Gold-layer questions before building a formal report.

They do not replace Gold modeling. They consume the same curated objects and
are another way to validate whether the Gold layer answers real business
questions.

`[UI DEMO]` Create a simple AI/BI dashboard from `gold.kpi_monthly`, then ask
Genie a question such as: "Which region has the highest revenue trend?"

## 12. Dataset readiness self-check

In [ ]:
required_objects = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
    f"{GOLD}.kpi_monthly",
]
for obj in required_objects:
    assert spark.catalog.tableExists(obj), f"Missing {obj}"

summary_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.fact_sales_dashboard_monthly").first()["n"]
detail_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.v_fact_sales_incremental").first()["n"]
assert summary_rows > 0, "Summary object is empty"
assert detail_rows > 0, "Detail object is empty"

print("[OK] Power BI semantic dataset readiness check passed")

## Summary and artefacts

Created or confirmed:

- `gold.fact_sales_dashboard_monthly`
- `gold.v_fact_sales_incremental`
- `gold.kpi_monthly`
- BI contract
- Import vs DirectQuery decision
- Power BI connector walkthrough

The next notebook makes this reportable dataset operational: performance,
cost guardrails, Lakeflow Jobs and DABs.